In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "telecom_guide.pdf"
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loader{len(pages)} pages from the PDF")
print("\n--- First page preview(first 500 chars)---")
print(pages[0].page_content[:500])

C:\Users\Sogo\AppData\Local\Temp\ipykernel_12460\3831774412.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\Sogo\Documents\agentic-ai\Agentic-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loader9 pages from the PDF

--- First page preview(first 500 chars)---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n","\n",".",""],
)
chunks =splitter.split_documents(pages)
len(chunks)

37

In [6]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

C:\Users\Sogo\Documents\agentic-ai\Agentic-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sogo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Sogo\Documents\agentic-ai\Agentic-AI\.venv\Lib\site-packages\huggingfa

Vector store ready. 37 vectors stored.


In [9]:
retriever =vector_store.as_retriever(search_kwargs={"k":3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved,1):
    print(f"---chunk {i}---")
    print(doc.page_content[:300])
    print()

---chunk 1---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

---chunk 2---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

---chunk 3---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT ="""\

You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system","SYSTEM_PROMPT"),
    ("human","{question}"),
])

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0,
    reasoning_format = "parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)
print("RAG chain assembled.")

RAG chain assembled.


In [20]:
question = "How does international roaming work and what charges should i expect?"
print(f"Q:{question}\n")
print("A:", chain.invoke(question))

Q:How does international roaming work and what charges should i expect?

A: ## International Roaming – What It Is and How It Works  

| Step | What Happens | Why It Matters |
|------|--------------|----------------|
| **1. Your phone leaves the home network** | As soon as you cross a border, your device stops seeing the cells of your home carrier and starts looking for any other network it can attach to. | The moment you’re out of coverage, you become a “roaming” user. |
| **2. The visited network “accepts” you** | The foreign operator checks a **roaming agreement** it has with your home carrier. If an agreement exists, it lets your SIM register on its network and assigns you a temporary “roaming” subscriber ID. | The agreement determines which services you can use (voice, SMS, data) and the wholesale rates the visited network will charge your home carrier. |
| **3. Your home carrier bills you** | Your home carrier receives a usage record from the visited network, adds its own markup, 